In [1]:
import pandas as pd

In [2]:
gsc_pages = pd.read_csv('../data/gsc_pages.csv')
gsc_queries = pd.read_csv('../data/gsc_queries.csv')

print(gsc_pages.head().assign(**{
    'Top pages': gsc_pages['Top pages'].head().str.replace(r'https?://[^/]+', '[client-site]', regex=True)
}))
print(f"gsc_queries loaded: {gsc_queries.shape[0]} rows, {gsc_queries.shape[1]} columns")

                                           Top pages  Clicks  Impressions  \
0                                     [client-site]/     585         4435   
1               [client-site]/blogs/paypal-in-nepal/     349        61503   
2                                [client-site]/zoom/     107         4076   
3  [client-site]/blogs/zoom-not-working-troublesh...     102        26600   
4  [client-site]/blogs/aws-in-2026-latest-service...      29        80547   

      CTR  Position  
0  13.19%      5.27  
1   0.57%      4.46  
2   2.63%     11.57  
3   0.38%      6.22  
4   0.04%      6.44  
gsc_queries loaded: 1000 rows, 5 columns


In [3]:
file_path = '../data/ga4_landing_page.csv'

with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

header_index = None
for i, line in enumerate(lines):
    if line.startswith("Landing page"):
        header_index = i
        break

print("GA4 header found at line:", header_index)

ga4 = pd.read_csv(file_path, skiprows=header_index)

print(ga4.head())

GA4 header found at line: 9
                                  Landing page  Sessions  Active users  \
0   /blogs/aws-in-2026-latest-services-updates       592           525   
1                                            /       572           470   
2  /blogs/simple-comparison-of-zoom-vs-discord       338           337   
3                       /blogs/paypal-in-nepal       257           215   
4  /blogs/amazon-bedrock-aws-ai-platform-guide       232           222   

   New users  Average engagement time per session  Key events  Total revenue  \
0        524                            17.701014           0              0   
1        465                            37.127622           0              0   
2        337                            14.482249           0              0   
3        215                            29.626459           0              0   
4        222                             7.564655           0              0   

   Session key event rate  
0                 

In [4]:
gsc_pages.columns = gsc_pages.columns.str.strip()
gsc_queries.columns = gsc_queries.columns.str.strip()
ga4.columns = ga4.columns.str.strip()

gsc_pages = gsc_pages.rename(columns={'Top pages': 'Page'})
ga4 = ga4.rename(columns={'Landing page': 'Page'})

print("GSC columns:", gsc_pages.columns.tolist())
print("GA4 columns:", ga4.columns.tolist())

GSC columns: ['Page', 'Clicks', 'Impressions', 'CTR', 'Position']
GA4 columns: ['Page', 'Sessions', 'Active users', 'New users', 'Average engagement time per session', 'Key events', 'Total revenue', 'Session key event rate']


In [5]:
def clean_url(col):
    return (
        col.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r'https?://[^/]+', '', regex=True)
        .str.rstrip('/')
    )

gsc_pages['Page'] = clean_url(gsc_pages['Page'])
ga4['Page'] = clean_url(ga4['Page'])

ga4 = ga4[ga4['Page'].str.startswith('/')]

print("GSC sample:", gsc_pages['Page'].head(3).tolist())
print("GA4 sample:", ga4['Page'].head(3).tolist())

GSC sample: ['', '/blogs/paypal-in-nepal', '/zoom']
GA4 sample: ['/blogs/aws-in-2026-latest-services-updates', '/blogs/simple-comparison-of-zoom-vs-discord', '/blogs/paypal-in-nepal']


In [6]:
page_metrics = pd.merge(
    gsc_pages,
    ga4,
    on='Page',
    how='inner'
)

print(f"Merged dataset: {page_metrics.shape[0]} rows, {page_metrics.shape[1]} columns")
print(page_metrics.head())

Merged dataset: 74 rows, 12 columns
                                         Page  Clicks  Impressions    CTR  \
0                      /blogs/paypal-in-nepal     349        61503  0.57%   
1                                       /zoom     107         4076  2.63%   
2  /blogs/zoom-not-working-troubleshoot-guide     102        26600  0.38%   
3  /blogs/aws-in-2026-latest-services-updates      29        80547  0.04%   
4                  /cloud-consulting-in-nepal      26         1431  1.82%   

   Position  Sessions  Active users  New users  \
0      4.46       257           215        215   
1     11.57        65            62         59   
2      6.22       140           134        133   
3      6.44       592           525        524   
4     11.50        53            25         22   

   Average engagement time per session  Key events  Total revenue  \
0                            29.626459           0              0   
1                            74.030769           0            

In [7]:
page_metrics = page_metrics.rename(columns={
    'Clicks': 'gsc_clicks',
    'Impressions': 'gsc_impressions',
    'CTR': 'gsc_ctr',
    'Position': 'gsc_position',
    'Sessions': 'ga_sessions',
    'Active users': 'ga_active_users',
    'New users': 'ga_new_users',
    'Average engagement time per session': 'ga_engagement_time',
    'Key events': 'ga_conversions'
})

page_metrics['gsc_ctr'] = page_metrics['gsc_ctr'].str.replace('%', '').astype(float) / 100

print(page_metrics.dtypes)

Page                          str
gsc_clicks                  int64
gsc_impressions             int64
gsc_ctr                   float64
gsc_position              float64
ga_sessions                 int64
ga_active_users             int64
ga_new_users                int64
ga_engagement_time        float64
ga_conversions              int64
Total revenue               int64
Session key event rate      int64
dtype: object


In [8]:
page_metrics['visibility_score'] = page_metrics['gsc_impressions'] * page_metrics['gsc_ctr']
page_metrics['click_to_session_ratio'] = page_metrics['ga_sessions'] / page_metrics['gsc_clicks']
page_metrics['engagement_per_session'] = page_metrics['ga_engagement_time']
page_metrics['engagement_efficiency'] = page_metrics['ga_engagement_time'] / page_metrics['ga_sessions']

print(page_metrics[['Page', 'visibility_score', 'click_to_session_ratio', 'engagement_efficiency']].head())

                                         Page  visibility_score  \
0                      /blogs/paypal-in-nepal          350.5671   
1                                       /zoom          107.1988   
2  /blogs/zoom-not-working-troubleshoot-guide          101.0800   
3  /blogs/aws-in-2026-latest-services-updates           32.2188   
4                  /cloud-consulting-in-nepal           26.0442   

   click_to_session_ratio  engagement_efficiency  
0                0.736390               0.115278  
1                0.607477               1.138935  
2                1.372549               0.262857  
3               20.413793               0.029900  
4                2.038462               0.224635  


In [9]:
page_metrics.sort_values('ga_sessions', ascending=False)[
    ['Page', 'gsc_clicks', 'gsc_impressions', 'ga_sessions', 'ga_engagement_time']
].head(10)

,Page,gsc_clicks,gsc_impressions,ga_sessions,ga_engagement_time
3,/blogs/aws-in-2026-latest-services-updates,29,80547,592,17.701014
27,/blogs/simple-comparison-of-zoom-vs-discord,2,3203,338,14.482249
0,/blogs/paypal-in-nepal,349,61503,257,29.626459
14,/blogs/amazon-bedrock-aws-ai-platform-guide,9,49523,232,7.564655
2,/blogs/zoom-not-working-troubleshoot-guide,102,26600,140,36.800000
45,/cybersecurity,0,145,87,4.954023
13,/blogs/complete-guide-to-aws-database,10,10061,78,6.948718
15,/blogs/what-is-aws-guide-amazon-web-services,8,19555,74,28.337838
1,/zoom,107,4076,65,74.030769
4,/cloud-consulting-in-nepal,26,1431,53,11.905660


In [10]:
def classify(row):
    if row['gsc_impressions'] > 20000 and row['ga_sessions'] < 100:
        return "High visibility / low engagement"
    elif row['ga_engagement_time'] > 30:
        return "High quality content"
    elif row['gsc_clicks'] > 100:
        return "High traffic driver"
    else:
        return "Normal"

page_metrics['category'] = page_metrics.apply(classify, axis=1)

page_metrics[['Page', 'category']]

,Page,category
0,/blogs/paypal-in-nepal,High traffic driver
1,/zoom,High quality content
2,/blogs/zoom-not-working-troubleshoot-guide,High quality content
3,/blogs/aws-in-2026-latest-services-updates,Normal
4,/cloud-consulting-in-nepal,Normal
...,...,...
69,/plans/zoom-webinar-plan,Normal
70,/faq/what-cloud-services-do-you-offer,Normal
71,/cloud-service,Normal
72,/faq/how-secure-is-your-cloud-infrastructure,Normal
